In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Data Preparation
df = sns.load_dataset('tips')
X = df.drop('total_bill', axis=1)
y = df['total_bill']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

# 2. Preprocessing
cat_cols = ["sex", "smoker", "day", "time"]
num_cols = ["tip", "size"]

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('std', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# 3. Applying Bagging
# We create 100 versions of the DecisionTree, each trained on a random subset
base_model = DecisionTreeRegressor(max_depth=5)

bagging_model = BaggingRegressor(
    estimator=base_model,
    n_estimators=100,      # Number of trees/models
    max_samples=0.8,       # Use 80% of data for each bag
    bootstrap=True,        # Use sampling with replacement
    oob_score=True,        # Enable Out-of-Bag validation
    n_jobs=-1,             # Use all CPU cores (parallel)
    random_state=1
)

# 4. Fit and Evaluate
bagging_model.fit(X_train, y_train)
y_pred = bagging_model.predict(X_test)

# 5. Results
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"OOB Score: {bagging_model.oob_score_:.4f}")

R2 Score: 0.4335
MSE: 46.9124
OOB Score: 0.4770
